# Realistic apples-to-apples serving benchmark

This fixes the serving-contract mismatch in the previous benchmark. Both models get persistent per-user state and are timed with the same CUDA-event harness.

**SASRec:** canonical ML-1M geometry (`d=64`, 2 layers, 1 head) with a fixed-size incremental KV ring using PyTorch SDPA. We report model-only, an *optimistic* `FREE-ANN-512` path where candidate discovery is given to SASRec for free, and dense exact scoring as a reference.

**SparseWalker:** Triton local Walker -> 2 real Triton temporal SWG reads -> temporal FFNs -> Triton sparse terminal retrieval. **No full-catalog dense matmul is used in the Walker path.**

The notebook reports eager p50/p95/p99 and then applies the same CUDA-Graph capture to both systems, which removes launch fragmentation symmetrically. It also prints per-user persistent-state memory.

Important: this is a speed comparison, not yet a quality-equivalent terminal-retrieval comparison. The 2-layer Walker NDCG was measured with dense scoring; sparse terminal support still needs distillation/evaluation on that checkpoint.

In [ ]:
import os, sys, subprocess, shutil, torch
REPO='/content/Sparsewalker'
BRANCH='agent/serving-speed-benchmark'
if os.path.exists(REPO): shutil.rmtree(REPO)
subprocess.run(['git','clone','-q','-b',BRANCH,'https://github.com/hanialshater/Sparsewalker-.git',REPO],check=True)
sys.path.insert(0,f'{REPO}/src')
for name in list(sys.modules):
    if name=='sparsewalker' or name.startswith('sparsewalker.'):
        del sys.modules[name]
assert torch.cuda.is_available(), 'GPU runtime required'
print('GPU',torch.cuda.get_device_name(0),'torch',torch.__version__,'bf16',torch.cuda.is_bf16_supported(),flush=True)
print('BRANCH',BRANCH,flush=True)

## Run

The default sweep is `history = 200 / 1000 / 10000` and catalog `3706 / 1M`. The most important output lines start with `APPLES`.


In [ ]:
from google.colab import drive
drive.mount('/content/drive',force_remount=False)
import runpy
SCRIPT=f'{REPO}/benchmarks/run_realistic_apples_serving.py'
sys.argv=[SCRIPT,'--lengths','200','1000','10000','--catalogs','3706','1000000','--beam','16','--hops','4','--degree','16','--terminal-degree','64']
print('REALISTIC APPLES BENCH START',flush=True)
runpy.run_path(SCRIPT,run_name='__main__')
print('REALISTIC APPLES BENCH END',flush=True)

## Compact saved result


In [ ]:
import json
from pathlib import Path
p=Path('/content/drive/MyDrive/sparsewalker_speed/realistic_apples_serving.json')
if p.exists():
    r=json.loads(p.read_text())
    for row in r['rows']:
        print('RESULT',{'history':row['history'],'catalog':row['catalog'],'speedups':row['cuda_graph_p50_speedups'],'state_memory':row['state_memory'],'graph_p50':{k:v['p50_us'] for k,v in row['cuda_graph'].items()}})
